# 03 · Measuring closeness: distance, dot product and cosine
Level L1 · Part 1 · builds on lesson 02

## Goal
By the end of this lesson you can compute Euclidean distance, dot product and cosine similarity
by hand and with NumPy, and say when they disagree about which recipe is "closest".

## The idea
In lesson 02 we saw that recipes which taste alike sit close together on the sweet/spicy map.
There are three common ways to get that number. **Euclidean distance** measures the straight
line between two points. The **dot product** (the multiply-and-add score from lesson 01) rewards
vectors that are big on the same features. **Cosine similarity** only asks whether two vectors
point in the same direction, ignoring how long they are.
Think of a city: Euclidean distance is "how far away is the shop", cosine is "is it in the same
direction from home", and the dot product mixes direction with "how far out it is".
They can disagree; **normalisation** makes them agree.

## Picture

```
 spicy
  0.8 |              o Mango salsa [0.6, 0.8]  <- same direction as Q, twice as long
      |            /                              (cosine = 1, biggest dot product)
  0.4 |        Q [0.3, 0.4]  "mildly spicy and a bit sweet"
      |     /  :
      |   /    : 0.3  <- straight-line gap (Euclidean distance)
  0.1 |  o     o Tomato soup [0.3, 0.1]   nearest point to Q
      | Caesar [0.1, 0.1]
  0.0 +----------------------o Lemon sorbet [0.8, 0] --o Brownies [0.95, 0] -- sweet

  sketch, not to scale. Tomato soup sits straight below Q: same sweetness, 0.3 less spicy.
  Think of each recipe as an arrow from [0, 0]: direction = which way, length = how strong
```

## Step by step

### 1. The question as a vector
Lesson 02 scored the five recipes on `[sweet, spicy]`. A question uses the **same features in
the same order**: "mildly spicy and a bit sweet" becomes `[0.3, 0.4]`. Now "which recipe is
closest?" is a question about two lists of numbers.

In [1]:
FEATURES = ["sweet", "spicy"]
recipe_vectors = {          # [sweet, spicy], same hand scores as lesson 02
    "Tomato soup":        [0.3, 0.1],
    "Mango salsa":        [0.6, 0.8],
    "Chocolate brownies": [0.95, 0.0],
    "Caesar salad":       [0.1, 0.1],
    "Lemon sorbet":       [0.8, 0.0],
}
query = [0.3, 0.4]          # "mildly spicy and a bit sweet"

for feature, value in zip(FEATURES, query, strict=True):
    print(f"query {feature:5} = {value}")

query sweet = 0.3
query spicy = 0.4


### 2. Euclidean distance by hand
The straight-line distance between two points is Pythagoras: subtract dimension by dimension,
square each difference, add them, take the square root. **Smaller means closer**; 0 means the
same point. Here it is for the query and Tomato soup.

In [2]:
import math

def euclidean(a, b):
    return math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b, strict=True)))

soup = recipe_vectors["Tomato soup"]
print("differences:", [round(x - y, 2) for x, y in zip(query, soup, strict=True)])
print(f"Euclidean distance query -> Tomato soup = {euclidean(query, soup):.3f}")

differences: [0.0, 0.3]
Euclidean distance query -> Tomato soup = 0.300


The differences are 0.0 in sweetness and 0.3 in spiciness, so the distance is exactly 0.3.

### 3. The dot product: lesson 01's score gets its name
Lesson 01 compared a question with each recipe by multiplying the matching scores and adding
them up. That calculation is the **dot product**. **Bigger means a better match**: it grows when
both vectors are high on the same features.

In [3]:
def dot(a, b):
    return sum(x * y for x, y in zip(a, b, strict=True))

print("products:", [round(x * y, 2) for x, y in zip(query, soup, strict=True)])
print(f"dot(query, Tomato soup) = {dot(query, soup):.3f}")

products: [0.09, 0.04]
dot(query, Tomato soup) = 0.130


0.3 × 0.3 + 0.4 × 0.1 = 0.09 + 0.04 = 0.13.

### 4. Length, then cosine similarity
A vector's **length** (also called its **norm**) is its Euclidean distance from `[0, 0]`.
**Cosine similarity** is the dot product divided by both lengths. Dividing removes the effect of
length, so only the direction is left. The result is between -1 and 1; with our scores, which
are never negative, it is between 0 and 1. 1 means "pointing exactly the same way".

In [4]:
def length(a):
    return math.sqrt(dot(a, a))

def cosine(a, b):
    return dot(a, b) / (length(a) * length(b))

print(f"length(query)       = {length(query):.3f}")
print(f"length(Tomato soup) = {length(soup):.3f}")
print(f"cosine(query, Tomato soup) = {cosine(query, soup):.3f}")

length(query)       = 0.500
length(Tomato soup) = 0.316
cosine(query, Tomato soup) = 0.822


The query `[0.3, 0.4]` has length 0.5 (a 3-4-5 triangle), Tomato soup about 0.316, so
cosine = 0.13 / (0.5 × 0.316) ≈ 0.822.

### 5. All five recipes at once with NumPy
Writing loops is fine for one pair. With a matrix of vectors (one row per recipe, as in
lesson 02), NumPy does every recipe in one line:
- `vectors @ q` is the dot product of every row with `q`.
- `np.linalg.norm(x, axis=1)` is the length of every row.
- `vectors - q` subtracts `q` from every row, so its row lengths are the Euclidean distances.

In [5]:
import numpy as np

names = list(recipe_vectors)
vectors = np.array([recipe_vectors[n] for n in names], dtype=np.float32)
q = np.array(query, dtype=np.float32)

dots = vectors @ q
euclid = np.linalg.norm(vectors - q, axis=1)
cosines = dots / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(q))

print(f"{'recipe':20} {'euclidean':>9} {'dot':>6} {'cosine':>7}")
for name, e, d, c in zip(names, euclid, dots, cosines, strict=True):
    print(f"{name:20} {e:9.3f} {d:6.3f} {c:7.3f}")

recipe               euclidean    dot  cosine
Tomato soup              0.300  0.130   0.822
Mango salsa              0.500  0.500   1.000
Chocolate brownies       0.763  0.285   0.600
Caesar salad             0.361  0.070   0.990
Lemon sorbet             0.640  0.240   0.600


The Tomato soup row matches the hand calculations: 0.300, 0.130, 0.822.

### 6. Rank by each measure: they disagree
To rank, sort distances **smallest first** and similarities **largest first**.
`np.argsort` returns row numbers in ascending order; `[::-1]` flips it to descending.

In [6]:
rankings = {
    "euclidean (small = close)": np.argsort(euclid),
    "dot       (big = close)":   np.argsort(dots)[::-1],
    "cosine    (big = close)":   np.argsort(cosines)[::-1],
}
for metric, order in rankings.items():
    print(f"{metric}: {[names[i] for i in order[:3]]}")

euclidean (small = close): ['Tomato soup', 'Caesar salad', 'Mango salsa']
dot       (big = close): ['Mango salsa', 'Chocolate brownies', 'Lemon sorbet']
cosine    (big = close): ['Mango salsa', 'Caesar salad', 'Tomato soup']


Three measures, three different top-3 lists:
- **Euclidean** picks Tomato soup: it is the nearest *point* (only 0.3 less spicy than asked).
- **Dot product** picks Mango salsa, then **Chocolate brownies**, which has no spice at all.
  The dot product grows with length, and brownies has a long vector (0.95 sweet), so "a lot of
  anything" scores well.
- **Cosine** gives Mango salsa a perfect 1.000 because `[0.6, 0.8]` is exactly 2 × `[0.3, 0.4]`:
  same direction, twice as strong. Caesar salad (0.990) comes second: `[0.1, 0.1]` points almost
  the same way, though it barely tastes of anything.

None is wrong: Euclidean compares amounts, cosine compares the balance of flavours, dot compares flavours and strength.

### 7. Normalisation: make every length 1
**Normalising** a vector means dividing it by its own length, so its length becomes 1
(a **unit vector**). Its direction does not change. Let's normalise every recipe and the query.

In [7]:
unit_vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
unit_q = q / np.linalg.norm(q)

print("lengths after normalising:", np.round(np.linalg.norm(unit_vectors, axis=1), 3))
print("unit query:", np.round(unit_q, 3))
print("unit Mango salsa:", np.round(unit_vectors[names.index("Mango salsa")], 3))

lengths after normalising: [1. 1. 1. 1. 1.]
unit query: [0.6 0.8]
unit Mango salsa: [0.6 0.8]


Mango salsa and the query become the very same unit vector `[0.6, 0.8]`: the "twice as strong"
information is gone. `keepdims=True` keeps the lengths as a column `(5, 1)`, so each row is divided
by its own length.

### 8. On unit vectors, all three agree
For unit vectors, the dot product **is** the cosine (both lengths are 1, so dividing changes
nothing). Euclidean distance also follows cosine exactly: distance² = 2 − 2 × cosine.
So all three rankings become the same.

In [8]:
unit_dots = unit_vectors @ unit_q
unit_euclid = np.linalg.norm(unit_vectors - unit_q, axis=1)

print(f"{'recipe':20} {'cosine':>7} {'unit dot':>8} {'unit eucl':>9} {'2-2cos':>7} {'eucl^2':>7}")
for i, name in enumerate(names):
    print(f"{name:20} {cosines[i]:7.3f} {unit_dots[i]:8.3f} {unit_euclid[i]:9.3f} "
          f"{2 - 2 * cosines[i]:7.3f} {unit_euclid[i] ** 2:7.3f}")

print("unit dot ranking:      ", [names[i] for i in np.argsort(unit_dots)[::-1]])
print("unit euclidean ranking:", [names[i] for i in np.argsort(unit_euclid)])

recipe                cosine unit dot unit eucl  2-2cos  eucl^2
Tomato soup            0.822    0.822     0.596   0.356   0.356
Mango salsa            1.000    1.000     0.000   0.000   0.000
Chocolate brownies     0.600    0.600     0.894   0.800   0.800
Caesar salad           0.990    0.990     0.142   0.020   0.020
Lemon sorbet           0.600    0.600     0.894   0.800   0.800
unit dot ranking:       ['Mango salsa', 'Caesar salad', 'Tomato soup', 'Lemon sorbet', 'Chocolate brownies']
unit euclidean ranking: ['Mango salsa', 'Caesar salad', 'Tomato soup', 'Chocolate brownies', 'Lemon sorbet']


The `cosine` and `unit dot` columns match, the last two columns match, and both rankings follow
the cosine ranking from step 6. (Brownies and sorbet tie at 0.600: both point straight along
"sweet", so their order inside the tie is arbitrary.) This is why many systems normalise vectors
once when storing them and then use the cheaper dot product: cosine results, with no division for
each stored vector at query time (only the query is normalised, once).

### 9. Which one should I use?
| Measure | Better match is | Cares about length? | Use when |
| --- | --- | --- | --- |
| Euclidean distance | smaller | yes | the size of each number matters (our hand-scored amounts) |
| Dot product | bigger | yes | vectors are already normalised, or length should count as "strength" |
| Cosine similarity | bigger | no | only the direction (the balance of features) matters |

For our hand-made scores, length means "how much flavour", so Euclidean gives the most sensible
answer: Tomato soup. From lesson 05 a model writes the vectors, and the rule changes to **use the
measure the model was trained for**: for most text embedding models that is cosine (or dot product
on normalised vectors). Vector databases ask you to pick one when you create a collection
(lesson 08).

## What just happened
- The question "mildly spicy and a bit sweet" became the vector `[0.3, 0.4]` (step 1).
- By hand, query vs Tomato soup gave Euclidean 0.300, dot 0.130, cosine 0.822 (steps 2–4); NumPy reproduced them for all five recipes in three lines (step 5).
- The rankings disagreed: Euclidean chose Tomato soup, dot chose Mango salsa then brownies, cosine gave Mango salsa a perfect 1.000 (step 6).
- Normalising made every length 1 and turned the query and Mango salsa into the same unit vector (step 7).
- On unit vectors, dot equals cosine and distance² equals 2 − 2 × cosine, so all three rankings matched (step 8).

## Common mistakes

**1. Comparing vectors with different dimensions.** The 3-D vectors from lesson 02 added "warm".
A 3-D query cannot be compared with the 2-D recipe matrix; NumPy refuses:

In [9]:
try:
    query_3d = np.array([0.3, 0.4, 0.8], dtype=np.float32)   # [sweet, spicy, warm]
    print("vectors:", vectors.shape, " query_3d:", query_3d.shape)
    vectors @ query_3d
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

vectors: (5, 2)  query_3d: (3,)
ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 3 is different from 2)


Fix: build the query from the **same feature list** as the matrix it is compared with.
Here that means the 3-D matrix from lesson 02 (sweet, spicy, plus lesson 01's warm scores).

In [10]:
FEATURES_3D = ["sweet", "spicy", "warm"]
warm = {"Tomato soup": 0.9, "Mango salsa": 0.1, "Chocolate brownies": 0.4,
        "Caesar salad": 0.1, "Lemon sorbet": 0.0}
vectors_3d = np.hstack([vectors, np.array([[warm[n]] for n in names], dtype=np.float32)])
print("vectors_3d:", vectors_3d.shape, " query_3d:", query_3d.shape)
print("dot product with Tomato soup:", round(float(vectors_3d[names.index("Tomato soup")] @ query_3d), 3))

vectors_3d: (5, 3)  query_3d: (3,)
dot product with Tomato soup: 0.85


**2. Cosine with an all-zero vector.** A recipe (or query) scored `[0, 0]`, "neither sweet nor
spicy", has length 0, and cosine divides by the length. First, the NumPy cosine from step 5:

In [11]:
zero = np.array([0.0, 0.0], dtype=np.float32)
with np.errstate(invalid="ignore"):   # silences NumPy's warning so the output stays tidy
    result = zero @ q / (np.linalg.norm(zero) * np.linalg.norm(q))
print("NumPy cosine with a zero vector:", result)

NumPy cosine with a zero vector: nan


Without the `np.errstate` line, NumPy prints `RuntimeWarning: invalid value encountered in
scalar divide` and carries on with `nan`. Now the plain-Python `cosine` from step 4:

In [12]:
try:
    plain_water = [0.0, 0.0]
    cosine(query, plain_water)
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

ZeroDivisionError: float division by zero


Python stops with `ZeroDivisionError`. The NumPy version is worse: it does not stop.
It returns `nan` ("not a number"), and `nan` sorts unpredictably, so it
quietly corrupts a ranking. A zero vector has no direction, so cosine is undefined for it.
Fix: check the lengths and fail with a clear message (or drop such vectors before indexing).

In [13]:
def safe_cosine(a, b):
    la, lb = length(a), length(b)
    if la == 0 or lb == 0:
        raise ValueError("cosine is undefined for a zero-length vector; check its scores")
    return dot(a, b) / (la * lb)

print(f"safe_cosine(query, Tomato soup) = {safe_cosine(query, soup):.3f}")
try:
    safe_cosine(query, plain_water)
except ValueError as err:
    print("caught:", err)

safe_cosine(query, Tomato soup) = 0.822
caught: cosine is undefined for a zero-length vector; check its scores


**3. Sorting a distance the wrong way.** Distances are "smaller is closer", similarities are
"bigger is closer". Sorting Euclidean distances largest-first quietly returns the *farthest*
recipes and nothing crashes:

In [14]:
order_wrong = np.argsort(euclid)[::-1]   # largest distance first
print("wrong (largest first): ", [names[i] for i in order_wrong[:3]])
print("right (smallest first):", [names[i] for i in np.argsort(euclid)[:3]])

wrong (largest first):  ['Chocolate brownies', 'Lemon sorbet', 'Mango salsa']
right (smallest first): ['Tomato soup', 'Caesar salad', 'Mango salsa']


The "wrong" line starts with Chocolate brownies, the recipe farthest from the query; the "right"
line starts with Tomato soup. A one-line sanity check (the top result must have the smallest
distance) turns this silent bug into a visible error:

In [15]:
try:
    closest = names[order_wrong[0]]
    assert euclid[order_wrong[0]] == euclid.min(), f"top result {closest} is not the nearest recipe"
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

AssertionError: top result Chocolate brownies is not the nearest recipe


It prints `AssertionError: top result Chocolate brownies is not the nearest recipe`. The fix is the
smallest-first order, `np.argsort(euclid)`: always name the direction when you sort, as `rankings`
did in step 6, and keep that one-line check in your tests.

## Try it
1. **Double the query.** Use `[0.6, 0.8]` ("twice as strong") and rank the recipes with all three
   measures. Which rankings change compared with step 6, and why do the others stay the same?
2. **Add warmth.** Use `vectors_3d` and the 3-D query `[0.3, 0.4, 0.8]` ("mildly spicy, a bit
   sweet, and warm"). Which recipe is top under each measure? Do they agree now?

In [16]:
# your code here

## Key terms
- **Euclidean distance** — the straight-line distance between two vectors; smaller means closer (also called L2 distance).
- **dot product** — multiply matching numbers of two vectors and add them up; bigger means a better match, and longer vectors score higher.
- **length (norm)** — how long a vector is: its Euclidean distance from the origin `[0, 0, ...]`.
- **cosine similarity** — the dot product divided by both lengths; compares direction only, from -1 to 1.
- **normalisation** — dividing a vector by its length so its length becomes 1; afterwards dot product equals cosine.
- **unit vector** — a vector of length 1.